# 04 — Synthetic ground-truth skeleton recovery

This notebook performs a **paired** validation of the historical `poly2graph` skeleton-to-graph path and the new sparse topology-aware junction-zone extractor. Every admissible synthetic volume is skeletonized exactly once, and the identical Lee skeleton is supplied to both graph extractors.

The controlled pathway is

$$G_{\rm true}\to F(G_{\rm true})\to V_{200}\to S_{\rm Lee}\to \widehat G. $$

Only connected, bridgeless, exactly trivalent ground-truth graphs are used. Yamada is called **only when the recovered graph has maximum degree $\le 3$**; this keeps the embedding-invariant check inside the degree regime used by the project.

The benchmark reports graph accuracy and timing separately. `SKIP-GEOM` means that the requested tube is too thick relative to the continuum embedding clearance and is therefore not counted as an admissible reconstruction case.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import statistics, sys, time
import networkx as nx
import numpy as np
import sympy as sp
from skimage.morphology import ball, dilation, skeletonize

ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'pyproject.toml').exists(): ROOT=ROOT.parent
if not (ROOT/'pyproject.toml').exists(): raise RuntimeError('Run inside the KnottedGraph checkout.')

import knotted_graph
from knotted_graph.core import contract_short_edges, remove_leaf_nodes, simplify_edges, smooth_edges
from knotted_graph.extraction import skeleton_image_to_graph
from knotted_graph.invariants.yamada.native import native_available, native_import_error
from knotted_graph.projection import compute_yamada_polynomial

print('Python executable:',sys.executable)
print('KnottedGraph:',Path(knotted_graph.__file__).resolve())
print('Native Yamada backend:',native_available())
print('Native import error:',native_import_error())

A=sp.Symbol('A')
BOUND=1.35
N=200
RADII=[1,2,3]
TRANSFORMS=['identity','rotate','affine']
CLEARANCE_FRACTION=0.40
JUNCTION_CONTRACT_VOXELS=2.5
SMOOTH_VOXELS=2.0
PROJECTION_SAMPLES=16
TIMING_REPEATS=3
DX=2*BOUND/(N-1)
print(f'N={N}, dx={DX:.6f}, radii={RADII}, transforms={TRANSFORMS}')


In [ ]:
@dataclass
class Case:
    name:str
    graph:nx.MultiGraph
    radius_cap:float

def normalize(X,scale=.72):
    X=np.asarray(X,float); X-=X.mean(0)
    return X*(scale/np.max(np.linalg.norm(X,axis=1)))

def embedded_graph(pos,edges):
    H=nx.MultiGraph()
    for n,p in pos.items(): H.add_node(n,pos=np.asarray(p,float))
    for u,v,P in edges: H.add_edge(u,v,pts=np.asarray(P,float))
    return H

def theta_case(name,bowed=False,n=500):
    t=np.linspace(0,1,n); x=-.72+1.44*t
    if bowed:
        C=[np.c_[x,-.58*np.sin(np.pi*t), .16*np.sin(2*np.pi*t)],
           np.c_[x, .10*np.sin(2*np.pi*t),-.10*np.sin(np.pi*t)],
           np.c_[x, .58*np.sin(np.pi*t),-.16*np.sin(2*np.pi*t)]]
    else:
        C=[np.c_[x,-.58*np.sin(np.pi*t),0*t],np.c_[x,0*t,0*t],np.c_[x,.58*np.sin(np.pi*t),0*t]]
    for P in C: P[0]=[-.72,0,0]; P[-1]=[.72,0,0]
    return Case(name,embedded_graph({'u':C[0][0],'v':C[0][-1]},[('u','v',P) for P in C]),.060)

def segdist(p1,q1,p2,q2):
    u=q1-p1; v=q2-p2; w=p1-p2
    a=u@u; b=u@v; c=v@v; d=u@w; e=v@w; D=a*c-b*b
    if D<1e-14: s=0.; t=np.clip(e/c if c>1e-14 else 0.,0,1)
    else: s=np.clip((b*e-c*d)/D,0,1); t=np.clip((a*e-b*d)/D,0,1)
    if a>1e-14: s=np.clip((b*t-d)/a,0,1)
    if c>1e-14: t=np.clip((b*s+e)/c,0,1)
    return float(np.linalg.norm(w+s*u-t*v))

def straight_clearance(G,P):
    E=list(G.edges()); best=np.inf
    for i,(u,v) in enumerate(E):
        for a,b in E[i+1:]:
            if {u,v}&{a,b}: continue
            best=min(best,segdist(P[u],P[v],P[a],P[b]))
    return best

def cubic_case(name,G,planar,seed,cap):
    G=nx.Graph(G)
    assert nx.is_connected(G) and not list(nx.bridges(G)) and all(d==3 for _,d in G.degree())
    if planar:
        ok,_=nx.check_planarity(G); assert ok
        p=nx.planar_layout(G); X=normalize([[p[n][0],p[n][1],0.] for n in G]); P={n:X[i] for i,n in enumerate(G)}
    else:
        P=None
        for trial in range(250):
            p=nx.spring_layout(G,dim=3,seed=seed+trial,iterations=700)
            X=normalize([p[n] for n in G]); Q={n:X[i] for i,n in enumerate(G)}
            if straight_clearance(G,Q)>.055: P=Q; break
        if P is None: raise RuntimeError(f'Could not find a clear 3D embedding for {name}')
    return Case(name,embedded_graph(P,[(u,v,np.linspace(P[u],P[v],100)) for u,v in G.edges()]),cap)

CASES=[
 theta_case('theta3_planar'),theta_case('theta3_bowed',True),
 cubic_case('K4',nx.complete_graph(4),True,11,.052),
 cubic_case('triangular_prism',nx.circular_ladder_graph(3),True,12,.045),
 cubic_case('cube',nx.cubical_graph(),True,13,.042),
 cubic_case('pentagonal_prism',nx.circular_ladder_graph(5),True,14,.035),
 cubic_case('dodecahedral',nx.dodecahedral_graph(),True,15,.027),
 cubic_case('K3_3',nx.complete_bipartite_graph(3,3),False,17,.032),
 cubic_case('petersen',nx.petersen_graph(),False,18,.030),
 cubic_case('heawood',nx.heawood_graph(),False,19,.024),
]
for c in CASES:
    ds=dict(c.graph.degree()); assert nx.is_connected(nx.Graph(c.graph)) and ds and set(ds.values())=={3}
    print(f'{c.name:20s} V={c.graph.number_of_nodes():2d} E={c.graph.number_of_edges():2d} degree=3')


In [ ]:
def Rxyz(a,b,c):
    a,b,c=np.deg2rad([a,b,c])
    Rx=np.array([[1,0,0],[0,np.cos(a),-np.sin(a)],[0,np.sin(a),np.cos(a)]])
    Ry=np.array([[np.cos(b),0,np.sin(b)],[0,1,0],[-np.sin(b),0,np.cos(b)]])
    Rz=np.array([[np.cos(c),-np.sin(c),0],[np.sin(c),np.cos(c),0],[0,0,1]])
    return Rz@Ry@Rx

def transform(name):
    if name=='identity': M,b=np.eye(3),np.zeros(3)
    elif name=='rotate': M,b=Rxyz(21,34,13),np.array([.04,-.03,.02])
    elif name=='affine':
        M=Rxyz(17,-23,31)@np.diag([1.08,.91,1.03])@np.array([[1,.13,0],[0,1,.09],[.05,0,1]])
        b=np.array([-.03,.04,-.02])
    else: raise ValueError(name)
    assert np.linalg.det(M)>0
    return M,b

def deform(G,name):
    M,b=transform(name); H=nx.MultiGraph()
    for n,d in G.nodes(data=True): H.add_node(n,pos=np.asarray(d['pos'])@M.T+b)
    for u,v,k,d in G.edges(keys=True,data=True): H.add_edge(u,v,pts=np.asarray(d['pts'])@M.T+b)
    return H

def trimmed(P,f=.15):
    P=np.asarray(P,float); n=max(1,int(round(f*len(P))))
    return P[n:-n] if 2*n<len(P) else P

def interior_sep(G):
    E=[(u,v,np.asarray(d['pts'],float)) for u,v,k,d in G.edges(keys=True,data=True)]
    best=np.inf
    for i,(u,v,P0) in enumerate(E):
        for a,b,Q0 in E[i+1:]:
            P,Q=(trimmed(P0),trimmed(Q0)) if {u,v}&{a,b} else (P0,Q0)
            for s in range(0,len(P),128):
                D2=np.sum((P[s:s+128,None,:]-Q[None,:,:])**2,axis=-1)
                best=min(best,float(np.sqrt(D2.min())))
    return best

def admissible(case,G,r):
    rw=r*DX; sep=interior_sep(G); limit=min(case.radius_cap,CLEARANCE_FRACTION*sep)
    return rw<=limit,rw,sep,limit

def resample(P,step):
    P=np.asarray(P,float); parts=[]
    for p,q in zip(P[:-1],P[1:]):
        n=max(2,int(np.ceil(np.linalg.norm(q-p)/step))+1); parts.append(np.linspace(p,q,n,endpoint=False))
    parts.append(P[-1:]); return np.vstack(parts)

def voxelize(G,r):
    V=np.zeros((N,N,N),bool)
    for _,_,_,d in G.edges(keys=True,data=True):
        P=resample(d['pts'],DX/3); I=np.rint((P+BOUND)/(2*BOUND)*(N-1)).astype(int); I=np.clip(I,0,N-1)
        V[I[:,0],I[:,1],I[:,2]]=1
    return dilation(V,footprint=ball(r))

def graph_stats(G):
    deg=[d for _,d in G.degree()]
    return {'V':G.number_of_nodes(),'E':G.number_of_edges(),'max_degree':max(deg,default=0),'degrees':sorted(deg)}

def to_world(H):
    H=nx.MultiGraph(H); o=np.array([-BOUND]*3,float)
    for _,d in H.nodes(data=True): d['pos']=o+DX*np.asarray(d['pos'],float)
    for _,_,_,d in H.edges(keys=True,data=True): d['pts']=o+DX*np.asarray(d['pts'],float)
    return H

def cleanup_baseline(H):
    H=remove_leaf_nodes(H); H=simplify_edges(H)
    H=contract_short_edges(H,min_length=JUNCTION_CONTRACT_VOXELS*DX,copy=False)
    H=remove_leaf_nodes(H); H=simplify_edges(H)
    return smooth_edges(H,epsilon=SMOOTH_VOXELS*DX,copy=False)

def cleanup_optimized(H):
    H=remove_leaf_nodes(H); H=simplify_edges(H)
    return smooth_edges(H,epsilon=SMOOTH_VOXELS*DX,copy=False)

def extract_baseline(S):
    return skeleton_image_to_graph(S,backend='poly2graph')

def extract_optimized(S):
    return skeleton_image_to_graph(S,backend='topology_aware',junction_hops=2,max_junction_degree=3)

def timed(fn,repeats=TIMING_REPEATS):
    values=[]; out=None
    for _ in range(repeats):
        t0=time.perf_counter(); out=fn(); values.append(time.perf_counter()-t0)
    return statistics.median(values),out

def abstract_ok(target,recovered):
    return nx.is_isomorphic(nx.MultiGraph(target),nx.MultiGraph(recovered))

def yamada(G):
    bad={n:d for n,d in G.degree() if d>3}
    if bad: raise ValueError(f'Yamada refused outside max-degree<=3 regime: {bad}')
    out=compute_yamada_polynomial(G,A,num_rotation_samples=PROJECTION_SAMPLES,crossing_warning_threshold=None,normalize=True,n_jobs=1,method='recursive',return_result=True)
    return sp.expand(out.polynomial),out.projection

def same(a,b): return sp.simplify(sp.together(sp.expand(a-b)))==0


In [ ]:
TARGETS={}
print('GROUND-TRUTH / PRE-VOXELIZATION CHECK')
for c in CASES:
    assert set(dict(c.graph.degree()).values())=={3}
    target,p=yamada(c.graph); TARGETS[c.name]=target
    print(f'TARGET {c.name:20s} V/E={c.graph.number_of_nodes()}/{c.graph.number_of_edges()} crossings={p.num_crossings:2d}')
    for t in TRANSFORMS:
        H=deform(c.graph,t); assert max(dict(H.degree()).values())<=3
        poly,_=yamada(H)
        if not same(poly,target): raise AssertionError(f'{c.name}/{t}: Yamada changed before voxelization')
print('PASS: all pre-voxelization deformations preserve normalized Yamada.')


In [ ]:
records=[]; skipped=0; admissible_count=0
baseline_extract_times=[]; optimized_extract_times=[]
baseline_recover_times=[]; optimized_recover_times=[]; skeleton_times=[]
warmed=False

def classify(target_graph,target_yamada,recovered):
    stats=graph_stats(recovered)
    if stats['max_degree']>3:
        return 'FAIL-DEG',stats,None
    if not abstract_ok(target_graph,recovered):
        return 'FAIL-GRAPH',stats,None
    poly,_=yamada(recovered)
    if not same(poly,target_yamada):
        return 'FAIL-EMBED',stats,poly
    return 'PASS',stats,poly

for c in CASES:
  for t in TRANSFORMS:
    G=deform(c.graph,t)
    for r in RADII:
      ok,rw,sep,limit=admissible(c,G,r)
      if not ok:
        skipped+=1
        print(f'SKIP-GEOM {c.name:20s} {t:8s} r={r} rw={rw:.4f} sep={sep:.4f} limit={limit:.4f}')
        continue
      admissible_count+=1
      V=voxelize(G,r)
      t0=time.perf_counter(); S=skeletonize(V,method='lee'); skeleton_times.append(time.perf_counter()-t0)
      if not np.any(S): raise RuntimeError(f'{c.name}/{t}/r={r}: empty skeleton')

      if not warmed:
        # Exclude one-time optional-import / Numba compilation effects.
        extract_baseline(S); extract_optimized(S); warmed=True

      b_ext,b_raw=timed(lambda: extract_baseline(S))
      o_ext,o_raw=timed(lambda: extract_optimized(S))
      baseline_extract_times.append(b_ext); optimized_extract_times.append(o_ext)

      t0=time.perf_counter(); B=cleanup_baseline(to_world(b_raw)); b_rec=time.perf_counter()-t0+b_ext
      t0=time.perf_counter(); O=cleanup_optimized(to_world(o_raw)); o_rec=time.perf_counter()-t0+o_ext
      baseline_recover_times.append(b_rec); optimized_recover_times.append(o_rec)

      b_status,b_stats,_=classify(c.graph,TARGETS[c.name],B)
      o_status,o_stats,_=classify(c.graph,TARGETS[c.name],O)
      records.append({'case':c.name,'transform':t,'r':r,'baseline':b_status,'optimized':o_status,'b_stats':b_stats,'o_stats':o_stats,'b_extract':b_ext,'o_extract':o_ext})
      print(f'{c.name:20s} {t:8s} r={r}  baseline={b_status:10s} {b_stats["V"]}/{b_stats["E"]}  optimized={o_status:10s} {o_stats["V"]}/{o_stats["E"]}  extract_ms={1e3*b_ext:.2f}->{1e3*o_ext:.2f}')

baseline_pass=sum(row['baseline']=='PASS' for row in records)
optimized_pass=sum(row['optimized']=='PASS' for row in records)
b_extract=statistics.median(baseline_extract_times); o_extract=statistics.median(optimized_extract_times)
b_recover=statistics.median(baseline_recover_times); o_recover=statistics.median(optimized_recover_times)
shared_skeleton=statistics.median(skeleton_times)
print('\nOVERALL PAIRED RESULT')
print(f'candidate points: {len(CASES)*len(TRANSFORMS)*len(RADII)}')
print(f'geometrically skipped: {skipped}')
print(f'admissible paired cases: {admissible_count}')
print(f'baseline exact end-to-end:  {baseline_pass}/{admissible_count} = {baseline_pass/admissible_count:.2%}')
print(f'optimized exact end-to-end: {optimized_pass}/{admissible_count} = {optimized_pass/admissible_count:.2%}')
print(f'median extraction: {1e3*b_extract:.3f} ms -> {1e3*o_extract:.3f} ms  speedup={b_extract/o_extract:.2f}x')
print(f'median graph recovery incl. cleanup: {1e3*b_recover:.3f} ms -> {1e3*o_recover:.3f} ms  speedup={b_recover/o_recover:.2f}x')
print(f'median shared Lee skeletonization: {1e3*shared_skeleton:.3f} ms')
print(f'median skeletonization+recovery: {1e3*(shared_skeleton+b_recover):.3f} ms -> {1e3*(shared_skeleton+o_recover):.3f} ms  speedup={(shared_skeleton+b_recover)/(shared_skeleton+o_recover):.2f}x')

print('\nOPTIMIZED FAILURES')
for row in records:
    if row['optimized']!='PASS': print(row)

assert admissible_count==len(records)>0
assert optimized_pass>=baseline_pass
assert optimized_pass/admissible_count>=0.95, 'optimized graph recovery did not reach the required accuracy regime'
assert optimized_pass-baseline_pass>=max(1,int(np.floor(0.20*admissible_count))), 'accuracy gain is not substantial'
assert o_extract<b_extract, 'topology-aware extraction is not faster than the historical extractor'
assert o_recover<b_recover, 'topology-aware graph recovery is not faster after cleanup'
print('\nPASS: topology-aware extraction substantially improves paired accuracy and runtime.')


## Interpretation

This is deliberately a paired test. A change in the reported recovery rate cannot be attributed to a different voxel volume, a different Lee skeleton, or a different admissibility filter: both extractors receive the same one-voxel skeleton.

The topology-aware algorithm differs at the skeleton-to-graph step. It identifies non-degree-two voxel neighborhoods, expands them into connected junction zones, contracts each physical junction zone once, traces the intervening degree-two chains, removes tiny zero-loop junction artifacts, and adaptively expands a junction by one additional graph hop only when a degree-at-most-three defect score indicates that the first contraction remains inconsistent.
